This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [ ]:
import great_expectations as gx
import logging

In [ ]:
from great_expectations_experimental.expectations.expect_queried_custom_query_to_return_num_rows import ExpectQueriedCustomQueryToReturnNumRows

In [ ]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [ ]:
context = gx.get_context(context_root_dir=gx_context_root_dir)
context.list_expectation_suites()

In [ ]:
import yaml

In [ ]:
from datetime import date,datetime

In [ ]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [ ]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [ ]:
datasource_config.get("project")

In [ ]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [ ]:
gx_datasource.get_asset_names()

In [ ]:
context.list_expectation_suite_names()

In [ ]:
TABLE_NAME = "messages_segmented_.3-0-0"
DUMMY_BATCH_DATE = '2023-02-01' # the date slice you want to run interactive expectations on

In [ ]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if TABLE_NAME in es and 'constraints' in es]:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: DUMMY_BATCH_DATE}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)



    gx_validator.expect_column_values_to_be_unique('msgid')
    gx_validator.expect_column_values_to_not_be_null('msgid')

    def timestamp_from_id_sql(id_column: str):
            return(f"""
        ((
            SELECT 
            TIMESTAMP(STRING_AGG(arr, '-'))
            FROM UNNEST(SPLIT({id_column}, '-')) AS arr WITH OFFSET as offset
            WHERE offset BETWEEN 1 and 3
        ))""")
    
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
            SELECT frag_id
            FROM {{active_batch}}
            WHERE frag_id IS NOT NULL
            AND DATE({timestamp_from_id_sql('frag_id')}) != DATE(timestamp)
        """}, value=0, meta={
                    "notes": {
                        "format": "markdown",
                        "content": "All timestamps in a frag_id should be on the same date as timestamp if frag_id is not null.",
                }
        })
    
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE frag_id IS NOT NULL
        AND LEFT(frag_id, STRPOS(frag_id, "-")-1) != ssvid
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "The `frag_id` should start with the `ssvid`.",
            }
    })
    
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE frag_id IS NOT NULL
        AND seg_id IS NOT NULL                                                                           
        AND DATE({timestamp_from_id_sql('frag_id')}) < DATE({timestamp_from_id_sql('seg_id')})
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "All timestamps in a seg_id should be on or before `frag_id`'s timestamp."
            }
    })


    # gx_validator.expect_column_distinct_values_to_be_in_set("source", ["spire", "orbcomm"],
    #  row_condition = 'col("timestamp") <= "2022-12-31"', condition_parser = "great_expectations__experimental__")
    gx_validator.expect_column_distinct_values_to_be_in_set("source", ["spire", "ais-listener", 'exactearth'],
     row_condition = 'col("timestamp") >= "2023-01-01"', condition_parser = "great_expectations__experimental__")
    # regex word start in BQ: https://stackoverflow.com/a/60728787/4166885
    gx_validator.expect_column_values_to_match_regex("type", "(?:^|\s)AIS.*")
    # gx_validator.expect_column_distinct_values_to_be_in_set("receiver_type", ["satellite", "terrestrial"],
    #  row_condition = 'col("timestamp") <= "2021-11-28"', condition_parser = "great_expectations__experimental__")
    gx_validator.expect_column_distinct_values_to_be_in_set("receiver_type", ["satellite", "terrestrial", "dynamic"],
     row_condition = 'col("timestamp") >= "2021-11-29"', condition_parser = "great_expectations__experimental__")

    gx_validator.expect_column_values_to_be_between(
        "lon", 
        min_value = -181, 
        max_value=181, 
        strict_min=True,
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("lon").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "lat", 
        min_value = -91, 
        max_value=91, 
        strict_min=True,
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("lat").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "course", 
        min_value = 0, 
        max_value=360, 
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("course").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "heading", 
        min_value = 0, 
        max_value=360, 
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("heading").notnull()'
    )

    gx_validator.expect_column_values_to_be_between(
        "speed", 
        min_value = 0, 
        max_value=102.3, 
        strict_max=True,
        condition_parser="great_expectations__experimental__", 
        row_condition = 'col("speed").notnull()'
    )

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE speed IS NOT NULL
        AND type = "AIS.27"
        AND
        (
            speed < 0
        OR
            speed >= 63
        )
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "Speed should be between 0 and 63 if type is AIS.27",
            }
    })

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE type IN ('AIS.5', 'AIS.19', 'AIS.21', 'AIS.24')
        AND (shipname = '@@@@@@@@@@@@@@@@@@@@')
    """}, value=0)

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE type IN ('AIS.5', 'AIS.24')
        AND shipname = '@@@@@@@'
    """}, value=0)

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE type IN ('AIS.5')
        AND destination = '@@@@@@@@@@@@@@@@@@@@'
    """}, value=0)
    
    # if some expectations fail "as expected" leaves this False
    # setting discard_failed_expectations=True could be useful when experimenting a lot
    gx_validator.save_expectation_suite(discard_failed_expectations=False)

In [ ]:
gx_validator.get_expectation_suite().expectations

In [ ]:
TIMESTAMP_COLUMN = 'timestamp'

In [ ]:
generated_lookups=f"""WITH 
generated_minute AS (SELECT * FROM UNNEST(GENERATE_ARRAY(0, 50)) col),
generated_10_minute_interval AS (SELECT * FROM UNNEST(GENERATE_ARRAY(0, 50, 10)) col),
generated_hour AS (SELECT * FROM UNNEST(GENERATE_ARRAY(0, 23)) col),
generated_minute_hour AS (
  SELECT
    CONCAT(
      LPAD(CAST(generated_hour.col AS STRING), 2, "0"), 
      ":", 
      LPAD(CAST(generated_minute.col AS STRING), 2, "0")
    ) col
  FROM generated_minute
  CROSS JOIN generated_hour),
generated_truncated_timestamp_minute AS(
  SELECT *
  FROM 
    UNNEST(
      GENERATE_TIMESTAMP_ARRAY(
        (SELECT TIMESTAMP_TRUNC(min(timestamp), minute) FROM {{active_batch}}), 
        (SELECT TIMESTAMP_TRUNC(max(timestamp), minute) FROM {{active_batch}}), 
        interval 1 minute
      )
    ) col
),
generated_truncated_timestamp_hour AS (
  SELECT *
  FROM 
  UNNEST(
    GENERATE_TIMESTAMP_ARRAY(
      (SELECT TIMESTAMP_TRUNC(min(timestamp), hour) FROM {{active_batch}}), 
      (SELECT TIMESTAMP_TRUNC(max(timestamp), hour) FROM {{active_batch}}), 
      interval 1 hour
      )
    ) col
),
active_batch_with_extra_columns AS (
SELECT
    True as in_active_batch,
    TIMESTAMP_TRUNC({TIMESTAMP_COLUMN}, second) timestamp_truncated_second,
    TIMESTAMP_TRUNC({TIMESTAMP_COLUMN}, minute) timestamp_truncated_minute,
    TIMESTAMP_TRUNC({TIMESTAMP_COLUMN}, hour) timestamp_truncated_hour,
    EXTRACT(minute from {TIMESTAMP_COLUMN}) extracted_minute,
    EXTRACT(hour from {TIMESTAMP_COLUMN}) extracted_hour,
    FORMAT_TIMESTAMP("%H:%M", timestamp) extracted_minute_hour,
    {TIMESTAMP_COLUMN} AS timestamp,
    *
FROM {{active_batch}}
)"""

In [ ]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if TABLE_NAME in es and 'anomalies' in es]:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: DUMMY_BATCH_DATE}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    {generated_lookups}
    SELECT col
    FROM active_batch_with_extra_columns

    -- the next 2 lines need to be adjusted
    FULL JOIN generated_minute_hour
    ON extracted_minute_hour = col
    --

    WHERE in_active_batch IS NULL
    ORDER BY col
        """}, value=0, meta={
                    "notes": {
                        "format": "markdown",
                        "content": "No gaps expected: For every minute of every hour of the day there should be a message.",
                }
        })

    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
    {generated_lookups}
    SELECT col
    FROM active_batch_with_extra_columns

    -- the next 2 lines need to be adjusted
    FULL JOIN generated_hour
    ON extracted_hour = col
    --

    WHERE in_active_batch IS NULL
    ORDER BY col
        """}, value=0, meta={
                    "notes": {
                        "format": "markdown",
                        "content": "No gaps expected: For every hour of the day there should be at least one message.",
                }
        })
    
    gx_validator.save_expectation_suite(discard_failed_expectations=False)

    gx_validator.expectation_suite()

In [ ]:
gx_validator.get_expectation_suite().expectations